# Baseline: RandomForest for tabular competition

A simple baseline notebook for tabular regression/classification style Kaggle tasks.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submition.csv")
train.head()

## Feature split and target detection

This cell shows a generic pattern for detecting target and preparing feature lists.

In [ ]:
TARGET_CANDIDATES = ["target", "label", "y", "price", "days", "occupied_days"]
target_col = next((c for c in TARGET_CANDIDATES if c in train.columns), train.columns[-1])

X = train.drop(columns=[target_col])
y = train[target_col]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("target:", target_col)
print("numeric:", numeric_features[:10])
print("categorical:", categorical_features[:10])

## Preprocessing pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Training and local evaluation

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

is_regression = y.dtype.kind in "fiu" and y.nunique() > 20
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
) if is_regression else RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])

pipeline.fit(X_train, y_train)
valid_pred = pipeline.predict(X_valid)

if is_regression:
    rmse = mean_squared_error(y_valid, valid_pred, squared=False)
    print({"rmse": rmse})
else:
    acc = accuracy_score(y_valid, valid_pred)
    print({"accuracy": acc})

## Inference and submission

In [ ]:
test_pred = pipeline.predict(test)
submission = sample_sub.copy()
submission.iloc[:, 1] = test_pred
submission.to_csv("submission_rf.csv", index=False)
submission.head()